# Prompt 25: Common Spark Errors and Failure Scenarios
## Databricks Certified Associate Developer for Apache Spark
### Topic 4 — Troubleshooting and Tuning Spark Applications (10%)

---

## Error Catalogue

| Error | Where it surfaces | Root cause | Fix |
|-------|-------------------|-----------|-----|
| `OutOfMemoryError` (driver) | Driver process crashes | `collect()` / `toPandas()` on large data | Avoid `collect()` on large datasets; use aggregation first |
| `OutOfMemoryError` (executor) | Task fails; retry then job fails | Partition too large for executor heap | `repartition()`, increase executor memory |
| `AnalysisException: cannot resolve column` | Job fails during analysis phase | Column doesn't exist, misspelled, or was dropped | Fix spelling; check schema with `printSchema()` |
| `AnalysisException: ambiguous column reference` | Job fails during analysis phase | Join created two columns with the same name | Alias DataFrames before join; `drop()` duplicate column |
| `Task not serializable` | Executor task fails | Lambda/UDF captures non-serializable object (SparkContext, DB connection) | Move reference inside the function; don't capture Spark objects |
| `StackOverflowError` | Driver or executor fails | Very long lineage chain without checkpointing | `df.checkpoint()` or `df.persist()` to break the chain |
| Data skew / straggler tasks | One task 10× slower | Skewed key — one partition has most of the data | AQE skew join; salting; `repartition(n, col)` |
| Schema mismatch | `AnalysisException` or wrong values | Actual schema differs from expected | Provide explicit schema; use `unionByName()` |
| `NullPointerException` in UDF | Task fails | UDF receives `None` input it doesn't guard against | Add `if s is not None` check inside UDF body |

---

## 1. OutOfMemoryError

### Driver OOM
Caused by calling `collect()`, `toPandas()`, or `take()` on a large DataFrame — all rows are moved to the driver's JVM heap.

```
java.lang.OutOfMemoryError: Java heap space
    at org.apache.spark.sql.execution.CollectLimitExec...
```

### Executor OOM
Caused by a single partition being too large for the executor heap — typically from a skewed key or reading a large file with too few partitions.

```
ExecutorLostFailure (executor 2 exited caused by one of the running tasks)
Reason: Container killed by YARN for exceeding memory limits.
```

---

## 2. AnalysisException — Column Not Found

```
pyspark.sql.utils.AnalysisException: cannot resolve 'custommer_id' given input columns: [customer_id, amount]
```

**Causes:**
- Column name is misspelled
- Column was dropped earlier in the pipeline
- Wrong DataFrame is being referenced
- Case mismatch (Spark SQL is case-insensitive but Python `col()` is case-sensitive by default)

---

## 3. AnalysisException — Ambiguous Column Reference

```
pyspark.sql.utils.AnalysisException: Reference 'id' is ambiguous, could be: id, id
```

Caused by joining two DataFrames that both have a column with the same name (other than the join key), and then referencing that column after the join.

---

## 4. Task Not Serializable

```
org.apache.spark.SparkException: Task not serializable
Caused by: java.io.NotSerializableException: org.apache.spark.SparkContext
```

Caused by a lambda or UDF closing over a non-serializable object (SparkContext, SparkSession, database connections, file handles). Spark serializes the function to send it to executors — if the captured object can't be serialized, the task fails.

---

## 5. StackOverflowError

```
java.lang.StackOverflowError
    at org.apache.spark.sql.catalyst.trees.TreeNode...
```

Caused by applying hundreds or thousands of transformations iteratively without materializing intermediate results, creating a lineage chain so deep that the JVM stack overflows when building the query plan.

---

## 6. Schema Mismatch

```
pyspark.sql.utils.AnalysisException: cannot resolve 'total_price' given input columns: [price, qty, ...]
```

Or silently wrong — columns in wrong order cause `union()` to misalign values.

---

## 7. NullPointerException in UDF

```
PythonException: An exception was thrown from a UDF:
AttributeError: 'NoneType' object has no attribute 'upper'
```

In [ ]:
# Cell 1: Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast, rand, lit, when, count, udf
from pyspark.sql.types import StringType, IntegerType, DoubleType, StructType, StructField
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName('CommonErrors') \
    .master('local[2]') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.driver.memory', '1g') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('SparkSession ready')

In [ ]:
# Cell 2: OutOfMemoryError — how to reproduce and fix

df = spark.range(0, 100_000).withColumn('value', rand())

# -------------------------------------------------------
# BAD: collect() pulls all rows into driver memory
# On a huge dataset this causes driver OOM
# -------------------------------------------------------
# rows = df.collect()   # DO NOT DO THIS on large DataFrames
# pd_df = df.toPandas() # Same risk

print('=== BAD patterns that cause driver OOM ===')
print('  df.collect()          — loads ALL rows into driver memory')
print('  df.toPandas()         — same, via PyArrow or row-by-row')
print('  df.take(1_000_000)    — still loads 1M rows into driver')

# -------------------------------------------------------
# GOOD: aggregate before collecting
# -------------------------------------------------------
summary = df.agg(F.count('id'), F.avg('value'), F.max('value')).collect()
print('\n=== GOOD: collect only the aggregated summary ===')
print(summary)   # small result — safe to collect

# -------------------------------------------------------
# GOOD: use show() for inspection instead of collect()
# -------------------------------------------------------
df.show(5)   # fetches only 5 rows to driver

# -------------------------------------------------------
# Executor OOM: partition too large
# -------------------------------------------------------
print('\n=== Executor OOM — diagnosis and fix ===')
print('Symptom: ExecutorLostFailure / container killed by YARN for exceeding memory limits')
print('Check: Spark UI Stages tab → shuffle read size per task; look for one task with huge input')
print('\nFix 1: Increase partitions before the heavy operation')
print('  df = df.repartition(200)   # more partitions = smaller per-partition data')
print('\nFix 2: Increase executor memory')
print('  spark.executor.memory=8g   (set in cluster config or SparkSession builder)')
print('\nFix 3: Reduce broadcast threshold to force SortMergeJoin for large tables')
print('  spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")')

In [ ]:
# Cell 3: AnalysisException — column not found and ambiguous reference

df_orders = spark.createDataFrame(
    [(1, 101, 250.0), (2, 102, 180.0), (3, 101, 90.0)],
    ['order_id', 'customer_id', 'amount']
)
df_customers = spark.createDataFrame(
    [(101, 'Alice', 'US'), (102, 'Bob', 'CA')],
    ['customer_id', 'name', 'country']
)

# -------------------------------------------------------
# AnalysisException: column not found
# -------------------------------------------------------
print('=== Column not found — trigger and fix ===')
try:
    df_orders.select(col('custommer_id')).show()  # typo: 'custommer_id'
except Exception as e:
    print(f'ERROR: {type(e).__name__}: {str(e)[:120]}')

# Fix: check available columns first
print('\nAvailable columns:', df_orders.columns)
df_orders.select(col('customer_id')).show(3)  # correct spelling

# -------------------------------------------------------
# AnalysisException: ambiguous column reference after join
# -------------------------------------------------------
print('\n=== Ambiguous column reference after join ===')
df_joined = df_orders.join(df_customers, on='customer_id', how='inner')
print('Columns after join:', df_joined.columns)
# customer_id appears once (join key merged), but no ambiguity here
# Ambiguity occurs when joining on a different column and both have 'id':

df_a = spark.createDataFrame([(1, 'x'), (2, 'y')], ['id', 'val_a'])
df_b = spark.createDataFrame([(1, 'p'), (2, 'q')], ['id', 'val_b'])
df_bad_join = df_a.join(df_b, df_a.id == df_b.id, how='inner')  # joins but leaves two 'id' columns
print('\nColumns with duplicate id:', df_bad_join.columns)
try:
    df_bad_join.select(col('id')).show()  # AMBIGUOUS — which 'id'?
except Exception as e:
    print(f'ERROR: {type(e).__name__}: {str(e)[:120]}')

# Fix 1: Use string join key so Spark merges the column
df_good_join = df_a.join(df_b, on='id', how='inner')  # only one 'id' in result
print('\nFix 1 — join on string key, no ambiguity:')
df_good_join.show()

# Fix 2: Alias DataFrames and qualify the column
df_a2 = df_a.alias('a')
df_b2 = df_b.alias('b')
df_qualified = df_a2.join(df_b2, col('a.id') == col('b.id'), how='inner') \
    .select(col('a.id'), col('a.val_a'), col('b.val_b'))
print('Fix 2 — qualified column reference:')
df_qualified.show()

# Fix 3: Drop the duplicate column after join
df_dropped = df_bad_join.drop(df_b.id)  # must reference the DataFrame, not a string
print('Fix 3 — drop duplicate column (must pass DataFrame.column, not string):')
df_dropped.select('id', 'val_a', 'val_b').show()

In [ ]:
# Cell 4: Task not serializable — reproduce and fix

df = spark.createDataFrame([('alice',), ('bob',), ('charlie',)], ['name'])

# -------------------------------------------------------
# BAD: UDF captures the SparkSession object (not serializable)
# -------------------------------------------------------
print('=== Task not serializable — BAD pattern ===')
# spark_ref = spark   # capturing SparkSession
# @udf(StringType())
# def bad_udf(name):
#     _ = spark_ref.sql('SELECT 1')   # captures spark — NOT serializable!
#     return name.upper() if name else None
# This will throw: org.apache.spark.SparkException: Task not serializable
print('BAD: UDF that closes over SparkSession/SparkContext/database connection')
print('     → spark serializes the function to send to executor')
print('     → SparkSession is not serializable → Task not serializable')

# -------------------------------------------------------
# GOOD: keep the UDF self-contained — no external references
# -------------------------------------------------------
print('\n=== Task not serializable — GOOD pattern ===')
@udf(StringType())
def good_udf(name):
    # No external captured objects — self-contained
    return name.upper() if name is not None else None

df.withColumn('upper_name', good_udf(col('name'))).show()

# -------------------------------------------------------
# Another common cause: database connection captured in closure
# -------------------------------------------------------
print('=== Database connection — BAD vs GOOD ===')
print('BAD:  capture connection in UDF closure (not serializable)')
print('GOOD: open the connection INSIDE the UDF or foreachBatch function')

# Illustrative pattern (no real DB):
def write_partition_to_db(iterator):
    # Open connection INSIDE the executor function — not captured from driver
    # conn = get_db_connection()   # opened per partition on the executor
    for row in iterator:
        pass   # process row
    # conn.close()

# df.rdd.foreachPartition(write_partition_to_db)  # correct pattern
print('Pattern: open DB connection inside foreachPartition, not in the driver scope')

In [ ]:
# Cell 5: StackOverflowError — long lineage and checkpoint fix

# -------------------------------------------------------
# StackOverflow reproduction: iterative transformations
# -------------------------------------------------------
print('=== StackOverflowError — long lineage ===')
print('BAD: applying 1000 iterative filter/withColumn transformations without caching')

# Illustrating the pattern (we stop at 50 to not actually overflow):
df_iter = spark.range(0, 100).withColumn('val', col('id').cast(DoubleType()))
for i in range(50):   # in real code this might be 500+ iterations
    df_iter = df_iter.withColumn('val', col('val') + lit(0.001))
# The lineage has 50+ transformations stacked — with 500+ this triggers StackOverflow

# Fix 1: persist() to break the lineage chain
print('\n=== Fix 1: persist() to break lineage ===')
df_persisted = spark.range(0, 100).withColumn('val', col('id').cast(DoubleType()))
for i in range(50):
    df_persisted = df_persisted.withColumn('val', col('val') + lit(0.001))
    if i % 10 == 9:   # break lineage every 10 iterations
        df_persisted.persist()
        df_persisted.count()  # materialise
        print(f'  Lineage broken at iteration {i+1} via persist()')

# Fix 2: checkpoint() — writes to disk and fully breaks lineage
print('\n=== Fix 2: checkpoint() breaks lineage at the filesystem level ===')
import tempfile, os
checkpoint_dir = os.path.join(tempfile.gettempdir(), 'spark_checkpoints')
spark.sparkContext.setCheckpointDir(checkpoint_dir)

df_ckpt = spark.range(0, 100).withColumn('val', col('id').cast(DoubleType()))
for i in range(30):
    df_ckpt = df_ckpt.withColumn('val', col('val') + lit(0.001))
    if i % 10 == 9:
        df_ckpt = df_ckpt.checkpoint()  # writes to disk; lineage reset to this point
        print(f'  Checkpoint written at iteration {i+1}')

df_ckpt.show(3)

df_persisted.unpersist()

In [ ]:
# Cell 6: Schema mismatch — union() vs unionByName()

# -------------------------------------------------------
# Schema mismatch with union() — columns must be same order
# -------------------------------------------------------
df_jan = spark.createDataFrame(
    [(1, 'Alice', 500.0), (2, 'Bob', 300.0)],
    ['customer_id', 'name', 'revenue']
)
df_feb = spark.createDataFrame(
    [(300.0, 3, 'Charlie'), (200.0, 4, 'Diana')],
    ['revenue', 'customer_id', 'name']   # columns in different order!
)

print('=== union() with mismatched column order — silently wrong! ===')
df_wrong = df_jan.union(df_feb)
df_wrong.show()
# revenue column of df_feb lands in customer_id position — no error, but wrong data!
print('Note: customer_id = 300.0 is WRONG — union() aligns by position, not name')

# Fix: use unionByName() — aligns by column name
print('\n=== unionByName() — correct alignment by column name ===')
df_correct = df_jan.unionByName(df_feb)
df_correct.show()

# -------------------------------------------------------
# Schema mismatch with missing columns
# -------------------------------------------------------
df_mar = spark.createDataFrame(
    [(5, 'Eve', 150.0, 'Premium')],
    ['customer_id', 'name', 'revenue', 'tier']  # extra column!
)

print('\n=== unionByName with missing columns — fix with allowMissingColumns=True ===')
try:
    df_jan.unionByName(df_mar).show()
except Exception as e:
    print(f'ERROR: {type(e).__name__}: {str(e)[:120]}')

# Fix: allowMissingColumns fills missing columns with null
df_allow = df_jan.unionByName(df_mar, allowMissingColumns=True)
df_allow.show()

# -------------------------------------------------------
# Schema mismatch on read — fix with explicit schema
# -------------------------------------------------------
print('\n=== Reading with explicit schema vs inferSchema ===')
print('inferSchema=True: may read integer as double, date as string, null columns as string')
print('Fix: always provide an explicit StructType schema for production reads')
print('''
schema = StructType([
    StructField('customer_id', IntegerType(), True),
    StructField('name',        StringType(),  True),
    StructField('revenue',     DoubleType(),  True),
])
df = spark.read.schema(schema).csv('/data/customers.csv')
''')

In [ ]:
# Cell 7: NullPointerException in UDF — reproduce and fix

df_with_nulls = spark.createDataFrame(
    [('alice',), ('bob',), (None,), ('charlie',)],
    ['name']
)

# -------------------------------------------------------
# BAD: UDF that doesn't handle None
# -------------------------------------------------------
print('=== NullPointerException in UDF — BAD pattern ===')
@udf(StringType())
def unsafe_upper(s):
    return s.upper()   # AttributeError: 'NoneType' object has no attribute 'upper'

try:
    df_with_nulls.withColumn('upper', unsafe_upper(col('name'))).show()
except Exception as e:
    print(f'ERROR: {type(e).__name__}: {str(e)[:200]}')

# -------------------------------------------------------
# GOOD: Always guard against None in UDF body
# -------------------------------------------------------
print('\n=== NullPointerException in UDF — GOOD pattern ===')
@udf(StringType())
def safe_upper(s):
    if s is None:
        return None   # propagate NULL; don't crash
    return s.upper()

df_with_nulls.withColumn('upper', safe_upper(col('name'))).show()

# -------------------------------------------------------
# Better: wrap UDF body in try/except to surface errors gracefully
# -------------------------------------------------------
print('\n=== UDF with try/except for surfacing errors ===')
@udf(StringType())
def robust_udf(s):
    try:
        if s is None:
            return None
        return s.strip().upper()
    except Exception as e:
        return f'ERROR:{str(e)}'   # return error info as string for debugging

df_with_nulls.withColumn('result', robust_udf(col('name'))).show()

# -------------------------------------------------------
# Prefer built-in functions — no UDF, no null issue
# -------------------------------------------------------
print('\n=== Best: use built-in F.upper() — no UDF needed, handles nulls natively ===')
df_with_nulls.withColumn('upper', F.upper(col('name'))).show()

spark.stop()
print('\nDone.')

## Real-World Scenarios

<details>
<summary>Scenario 1: Driver OOM when a data analyst calls collect() on a 50GB table</summary>

**Situation:**
An analyst wants to explore a 50GB events table by converting it to a pandas DataFrame. They run:
```python
df = spark.read.parquet('/data/events')
pandas_df = df.toPandas()   # attempts to move 50GB to driver
```
The driver crashes with `java.lang.OutOfMemoryError: Java heap space`.

**Fixes (in order of preference):**
```python
# Option 1: Sample a small fraction first
sample_df = df.sample(fraction=0.001).toPandas()  # ~50MB — safe

# Option 2: Aggregate in Spark, collect only the summary
summary = df.groupBy('event_type').agg(
    F.count('*').alias('cnt'),
    F.avg('duration').alias('avg_dur')
).toPandas()   # small result — safe

# Option 3: Filter to a small subset before collecting
small_df = df.filter(col('date') == '2026-04-22').limit(10000).toPandas()
```

**Outcome:** No OOM. The analyst gets the data they need without moving 50GB to the driver.

**Exam Sub-topic:** Driver OOM; `collect()` / `toPandas()` risk; sampling and aggregation as fixes
</details>

<details>
<summary>Scenario 2: AnalysisException after dropping a column mid-pipeline</summary>

**Situation:**
A pipeline drops the `customer_id` column for anonymisation early in the chain, then tries to use it again in a join further downstream:

```python
df_clean = df_raw \
    .drop('customer_id') \
    .withColumn('hashed_id', F.sha2(col('email'), 256))

# Much later in the pipeline:
df_result = df_clean.join(df_prefs, on='customer_id')  # ERROR!
# AnalysisException: cannot resolve 'customer_id' given input columns: [email, hashed_id, ...]
```

**Diagnosis:** Run `df_clean.printSchema()` — `customer_id` is missing.

**Fix:**
```python
# Join BEFORE dropping the column, or join on hashed_id
df_result = df_raw.join(df_prefs, on='customer_id') \
    .drop('customer_id') \
    .withColumn('hashed_id', F.sha2(col('email'), 256))
```

**Exam Sub-topic:** `AnalysisException` column not found; `printSchema()` for diagnosis; transformation ordering
</details>

<details>
<summary>Scenario 3: Task not serializable — UDF captures a logging object</summary>

**Situation:**
A developer creates a Python `logging.Logger` at module scope and uses it inside a UDF:

```python
import logging
logger = logging.getLogger('my_app')   # captured by closure — may not serialize

@udf(StringType())
def process(value):
    logger.info(f'Processing {value}')   # captures logger from driver scope
    return value.upper() if value else None
```

Depending on the logging handlers configured, this causes `Task not serializable` on executors.

**Fix:**
```python
@udf(StringType())
def process(value):
    import logging   # import INSIDE the UDF — creates fresh logger per executor
    logger = logging.getLogger('my_app')
    logger.info(f'Processing {value}')
    return value.upper() if value else None
```

**Rule:** Any import or object creation that needs to happen on the executor should be inside the function, not captured from the driver scope.

**Exam Sub-topic:** Task not serializable; UDF serialization; keeping UDFs self-contained
</details>

<details>
<summary>Scenario 4: Silent data corruption from union() with mismatched column order</summary>

**Situation:**
Two teams produce monthly DataFrames. Team A produces `[user_id, spend, country]`; Team B produces `[country, user_id, spend]`. A combining job uses `union()` and the resulting analysis shows users with IDs like `'US'` and `'CA'` — clearly wrong.

```python
# BAD: union() aligns by position
combined = df_team_a.union(df_team_b)   # country values land in user_id column!

# GOOD: unionByName() aligns by name
combined = df_team_a.unionByName(df_team_b)
```

**Why it's dangerous:** `union()` raises no error — it silently misaligns columns. The data looks valid but is wrong. This is particularly hard to catch in automated pipelines.

**Diagnosis:** Add an assertion after the union:
```python
assert combined.filter(col('user_id').rlike('[A-Z]{2}')).count() == 0, \
    'user_id contains country codes — likely column mismatch'
```

**Exam Sub-topic:** `union()` vs `unionByName()`; schema mismatch; silent data corruption
</details>

<details>
<summary>Scenario 5: StackOverflowError in iterative ML feature engineering</summary>

**Situation:**
A feature engineering pipeline applies 300 `withColumn()` calls in a loop to compute lagged features:

```python
df_features = df_raw
for lag in range(1, 301):
    df_features = df_features.withColumn(
        f'lag_{lag}',
        F.lag('value', lag).over(window_spec)
    )
# df_features.show()  → StackOverflowError when Spark tries to build the 300-deep plan
```

**Fix — break lineage every N iterations:**
```python
import tempfile, os
spark.sparkContext.setCheckpointDir(tempfile.mkdtemp())

df_features = df_raw
for lag in range(1, 301):
    df_features = df_features.withColumn(
        f'lag_{lag}',
        F.lag('value', lag).over(window_spec)
    )
    if lag % 50 == 0:                    # break lineage every 50 iterations
        df_features = df_features.checkpoint()   # writes to disk, resets lineage

df_features.show(3)
```

**Alternative:** use `persist()` instead of `checkpoint()` if you don't want disk writes — still breaks the in-memory plan tree.

**Exam Sub-topic:** `StackOverflowError`; `checkpoint()` vs `persist()` for lineage breaking; iterative transformations
</details>

## Exam Cheat Sheet

| Error | Cause | Fix |
|-------|-------|-----|
| Driver OOM | `collect()` / `toPandas()` on large data | Aggregate first; use `show()`; `sample()` |
| Executor OOM | Partition too large | `repartition(n)`; increase `spark.executor.memory` |
| `AnalysisException: cannot resolve 'X'` | Column misspelled / dropped | `printSchema()`; fix spelling; check ordering |
| `AnalysisException: Reference 'X' is ambiguous` | Join creates two columns with same name | Join `on='col'` (string); alias DataFrames; `drop(df.col)` |
| `Task not serializable` | UDF closes over SparkContext / non-serializable object | Move captured object inside UDF |
| `StackOverflowError` | Very long lineage chain | `df.checkpoint()` or `df.persist()` periodically |
| Data skew / straggler | One key dominates data | AQE skew join; salting; `repartition(n, col)` |
| Silent wrong data from `union()` | Column order mismatch | Use `unionByName()` |
| `unionByName` fails with missing columns | One DataFrame has extra column | `unionByName(df, allowMissingColumns=True)` |
| `NullPointerException` in UDF | UDF doesn't guard against `None` | Add `if s is None: return None` |
| `NullPointerException` — better fix | Use built-in functions | Built-ins handle `NULL` natively; avoid UDF |
| Schema mismatch on read | `inferSchema` gets types wrong | Provide explicit `StructType` schema |
| How to reproduce skew | Check Spark UI Tasks tab | Compare median vs max task duration |
| `checkpoint()` vs `persist()` | checkpoint writes to disk, breaks full lineage | Use checkpoint for very long chains |
| `drop(df.col)` vs `drop('col')` | `drop('col')` drops ALL columns named 'col' | `drop(df.col)` drops only that DataFrame's column |

---

## Practice Q&A

<details>
<summary>Q1: What is the difference between driver OOM and executor OOM, and how do you fix each?</summary>

**Driver OOM:** The driver process runs out of JVM heap memory. Caused by calling `collect()`, `toPandas()`, or `take(N)` on a large dataset — all rows are moved to the driver.
- Fix: aggregate in Spark before collecting; use `show()` for inspection; `sample()` for exploration.

**Executor OOM:** An executor's JVM heap runs out during task execution. Caused by a single partition being too large — from data skew or reading a large file with too few partitions.
- Fix: `df.repartition(n)` to split data into smaller partitions; increase `spark.executor.memory`; reduce `spark.sql.autoBroadcastJoinThreshold` to avoid broadcasting large tables.
</details>

<details>
<summary>Q2: After joining two DataFrames, you reference column 'id' and get an AnalysisException about ambiguity. What are three ways to fix this?</summary>

**A:**
1. **String join key** — `df_a.join(df_b, on='id', how='inner')` — Spark merges the column into one.
2. **Alias and qualify** — `df_a.alias('a').join(df_b.alias('b'), col('a.id') == col('b.id')).select(col('a.id'), ...)`
3. **Drop the duplicate** — `df_joined.drop(df_b.id)` — note: must pass `df_b.id` (the Column object), not the string `'id'`, otherwise ALL columns named `'id'` are dropped.
</details>

<details>
<summary>Q3: Why does union() produce silently wrong results when column order differs, and what is the fix?</summary>

**A:** `union()` aligns DataFrames by **column position**, not by name. If the column order differs between the two DataFrames, values end up in the wrong columns — no error is raised. The result looks valid but contains incorrect data.

Fix: use **`unionByName()`** which aligns by column name. If one DataFrame has extra columns that the other lacks, add `allowMissingColumns=True` to fill missing columns with `NULL`.
</details>

<details>
<summary>Q4: A UDF fails with AttributeError: 'NoneType' object has no attribute 'strip'. How do you fix it, and what is the preferred alternative?</summary>

**A:** The UDF received a `NULL` input value, which becomes `None` in Python. The UDF does not check for `None` before calling `.strip()`.

**Fix the UDF:**
```python
@udf(StringType())
def safe_strip(s):
    return s.strip() if s is not None else None
```

**Preferred alternative:** Use the built-in `F.trim(col('s'))` — built-in functions handle `NULL` natively, run at JVM speed with Catalyst optimisation, and require no null guard.
</details>

<details>
<summary>Q5: What causes Task not serializable and what is the rule to prevent it?</summary>

**A:** Spark serializes the task function (UDF, lambda, or `foreachPartition` function) to send it to executor processes. If the function **closes over** (captures) an object that cannot be serialized — such as a `SparkSession`, `SparkContext`, database connection, file handle, or certain Python objects — the serialization fails with `Task not serializable`.

**Rule:** A UDF or distributed function must be **self-contained**. Any object it needs (imports, connections, loggers) must be **created inside the function**, not captured from the driver scope. Never reference `spark` or `sc` inside a UDF.
</details>